# 🛡️ NariRaksha — Indian Women's Safety SLM
> **Fine-tune Qwen2.5-3B on Colab T4 using Unsloth · Publish to Hugging Face**
>
> Run cells top-to-bottom. Each cell is self-contained with clear status output.
> - **GPU required**: Runtime → Change runtime type → T4 GPU
> - **Total time**: ~45 min for 1000 steps on T4
> - **No API key needed**: Uses built-in mock data generator

## ⚙️ Cell 1 — Install (run once per session)

In [ ]:
# Install Unsloth — handles ALL CUDA/quantization internally.
# We deliberately do NOT install bitsandbytes separately;
# Unsloth bundles a compatible version automatically.
import subprocess, sys

def _run(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.stdout: print(result.stdout[-2000:])
    if result.returncode != 0 and result.stderr: print('STDERR:', result.stderr[-500:])
    return result

print('Installing Unsloth ...')
_run('pip install unsloth -q')

print('Installing supporting libraries ...')
_run('pip install datasets huggingface_hub pyyaml tqdm -q')

print('\n✅ Installation complete — no restart needed.')

## 📦 Cell 2 — Clone / pull the repository

In [ ]:
import os, subprocess

REPO_URL = 'https://github.com/Vikhram-Labs/NariRaksha.git'
REPO_DIR = '/content/NariRaksha'

if os.path.isdir(REPO_DIR):
    print('Repo already cloned — pulling latest changes ...')
    subprocess.run('git pull', shell=True, cwd=REPO_DIR)
else:
    print('Cloning repo ...')
    subprocess.run(f'git clone {REPO_URL} {REPO_DIR}', shell=True)

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print(f'\n✅ Working directory: {os.getcwd()}')

## 🔑 Cell 3 — Configuration (edit these values)

In [ ]:
# ─────────────────────────────────────────────
# ✏️  Edit these settings before running Cell 4+
# ─────────────────────────────────────────────

# Model to fine-tune (all tested on Colab T4)
MODEL_ID = 'Qwen/Qwen2.5-3B-Instruct'  # Options: 'unsloth/Phi-3-mini-4k-instruct', 'unsloth/gemma-2-2b-it'

# Dataset
DATASET_SIZE    = 1000   # How many training examples to generate
OPENAI_API_KEY  = ''     # Optional: set for GPT-3.5 generated data; leave '' for mock data

# Training
MAX_STEPS       = 500    # Increase for better quality (1000–2000 recommended)
LORA_R          = 16
BATCH_SIZE      = 2
GRAD_ACCUM      = 8      # Effective batch = BATCH_SIZE × GRAD_ACCUM = 16
LEARNING_RATE   = 2e-4
MAX_SEQ_LEN     = 1024

# Output
OUTPUT_DIR      = '/content/NariRaksha/models/nariraksha-ft'
DATASET_PATH    = '/content/NariRaksha/datasets/synthetic/nariraksha_synthetic.jsonl'

# Hugging Face publishing (leave blank to skip)
HF_TOKEN        = ''     # Your HF write token (Settings → Access Tokens)
HF_MODEL_REPO   = ''     # e.g. 'your-username/NariRaksha-Qwen2.5-3B'
HF_DATASET_REPO = ''     # e.g. 'your-username/NariRaksha-100K'

print('✅ Configuration saved.')
print(f'   Model     : {MODEL_ID}')
print(f'   Dataset   : {DATASET_SIZE} examples')
print(f'   Steps     : {MAX_STEPS}')
print(f'   HF publish: {"enabled" if HF_TOKEN else "disabled (HF_TOKEN is blank)"}')

## 📊 Cell 4 — Generate training dataset

In [ ]:
import os, sys
sys.path.insert(0, '/content/NariRaksha')
from nariraksha.data import generate_dataset

os.makedirs(os.path.dirname(DATASET_PATH), exist_ok=True)

# Count existing examples
existing = 0
if os.path.exists(DATASET_PATH):
    with open(DATASET_PATH) as f:
        existing = sum(1 for line in f if line.strip())

print(f'Existing examples: {existing}')
need = max(0, DATASET_SIZE - existing)

if need > 0:
    print(f'Generating {need} new examples ...')
    use_mock = (OPENAI_API_KEY == '')
    generate_dataset(
        target=need,
        output_path=DATASET_PATH,
        use_mock=use_mock,
        openai_api_key=OPENAI_API_KEY or None,
    )
else:
    print(f'✅ Already have {existing} examples — skipping generation.')

# Final count
with open(DATASET_PATH) as f:
    total = sum(1 for line in f if line.strip())
print(f'\n✅ Dataset ready: {total} total examples at {DATASET_PATH}')

## 🤖 Cell 5 — Load model with Unsloth

In [ ]:
from unsloth import FastLanguageModel
import torch

print(f'Loading {MODEL_ID} ...')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,       # auto: bfloat16 on Ampere+, float16 on T4
    load_in_4bit=True,  # 4-bit quantisation — no bitsandbytes needed directly
)

# Apply LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_R * 2,
    lora_dropout=0.05,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    bias='none',
    use_gradient_checkpointing='unsloth',  # saves VRAM on T4
    random_state=42,
)
model.print_trainable_parameters()
print('\n✅ Model and LoRA adapters ready.')

## 🏋️ Cell 6 — Train

In [ ]:
import json, os, torch
from datasets import Dataset
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported
from nariraksha.data import load_and_format

# Load and format dataset
print('Loading dataset ...')
rows = load_and_format(DATASET_PATH)
print(f'Loaded {len(rows)} formatted examples.')

if len(rows) < 10:
    raise RuntimeError('Dataset too small — run Cell 4 first.')

hf_dataset = Dataset.from_list(rows)

# Training arguments
os.makedirs(OUTPUT_DIR, exist_ok=True)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=hf_dataset,
    dataset_text_field='text',
    max_seq_length=MAX_SEQ_LEN,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        warmup_steps=50,
        max_steps=MAX_STEPS,
        learning_rate=LEARNING_RATE,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=25,
        optim='adamw_8bit',
        weight_decay=0.01,
        lr_scheduler_type='cosine',
        seed=42,
        output_dir=OUTPUT_DIR,
        save_steps=100,
        report_to='none',
    ),
)

print(f'\nStarting training for {MAX_STEPS} steps ...')
print(f'Effective batch size: {BATCH_SIZE * GRAD_ACCUM}')
train_stats = trainer.train()

print(f'\n✅ Training complete!')
print(f'   Loss : {train_stats.training_loss:.4f}')
print(f'   Steps: {train_stats.global_step}')

## 💾 Cell 7 — Save model locally

In [ ]:
print(f'Saving LoRA adapters to {OUTPUT_DIR} ...')
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print('\n✅ Model saved.')

# Also save as merged float16 model (optional — needed for GGUF/Ollama)
MERGED_DIR = OUTPUT_DIR + '-merged-fp16'
print(f'\nSaving merged float16 model to {MERGED_DIR} ...')
model.save_pretrained_merged(
    MERGED_DIR, tokenizer,
    save_method='merged_16bit',
)
print('✅ Merged model saved.')

## 🧪 Cell 8 — Quick inference test

In [ ]:
from unsloth import FastLanguageModel
import json

FastLanguageModel.for_inference(model)  # enable 2x faster inference

test_scenario = (
    "A 24-year-old woman in Pune has been receiving threatening messages "
    "from an ex-partner on Instagram. He is threatening to share private photographs "
    "unless she meets him. She is terrified and does not know what to do."
)

from nariraksha.data import SYSTEM_PROMPT
prompt = (
    f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
    f"<|im_start|>user\nAnalyse this safety scenario and provide your assessment:\n\n"
    f"{test_scenario}<|im_end|>\n"
    f"<|im_start|>assistant\n"
)

inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
outputs = model.generate(
    **inputs,
    max_new_tokens=512,
    temperature=0.1,
    do_sample=True,
)

response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
print('=== NariRaksha Output ===')
try:
    parsed = json.loads(response)
    print(json.dumps(parsed, indent=2, ensure_ascii=False))
except Exception:
    print(response)

## 🚀 Cell 9 — Publish to Hugging Face
> Set `HF_TOKEN`, `HF_MODEL_REPO`, and `HF_DATASET_REPO` in Cell 3 before running.

In [ ]:
if not HF_TOKEN:
    print('⚠️  HF_TOKEN is blank — skipping publish. Set it in Cell 3 to publish.')
else:
    from huggingface_hub import HfApi, login
    from datasets import load_dataset as hf_load

    login(token=HF_TOKEN)
    api = HfApi()

    # ── Publish LoRA adapter model ──────────────────────────────────────────
    if HF_MODEL_REPO:
        print(f'\nPublishing LoRA adapters → {HF_MODEL_REPO} ...')
        api.create_repo(HF_MODEL_REPO, repo_type='model', exist_ok=True)
        model.push_to_hub(HF_MODEL_REPO, token=HF_TOKEN)
        tokenizer.push_to_hub(HF_MODEL_REPO, token=HF_TOKEN)

        # Write a model card
        card = f"""---
language:
- en
- hi
- ta
tags:
- nariraksha
- safety
- women-safety
- indian-law
- qlora
- unsloth
base_model: {MODEL_ID}
license: apache-2.0
---

# NariRaksha — Indian Women's Safety SLM

Fine-tuned **{MODEL_ID}** (QLoRA, 4-bit) on the NariRaksha dataset of
Indian women's safety scenarios. Provides structured safety assessments
including risk type, severity, legal context (BNS/IT Act), and
recommended actions.

## Quick usage

```python
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    '{HF_MODEL_REPO}', load_in_4bit=True
)
FastLanguageModel.for_inference(model)
```

## Training
- **Base model**: `{MODEL_ID}`
- **Method**: QLoRA (r={LORA_R}, α={LORA_R*2})
- **Steps**: {MAX_STEPS}
- **Framework**: [Unsloth](https://github.com/unslothai/unsloth)

## Dataset
See [{HF_DATASET_REPO or 'NariRaksha-100K'}](https://huggingface.co/datasets/{HF_DATASET_REPO or 'NariRaksha/NariRaksha-100K'})
"""
        with open(f'{OUTPUT_DIR}/README.md', 'w') as f:
            f.write(card)
        api.upload_file(
            path_or_fileobj=f'{OUTPUT_DIR}/README.md',
            path_in_repo='README.md',
            repo_id=HF_MODEL_REPO, repo_type='model', token=HF_TOKEN,
        )
        print(f'  ✅ Model published → https://huggingface.co/{HF_MODEL_REPO}')
    else:
        print('  ⚠️  HF_MODEL_REPO blank — skipping model publish.')

    # ── Publish dataset ─────────────────────────────────────────────────────
    if HF_DATASET_REPO:
        print(f'\nPublishing dataset → {HF_DATASET_REPO} ...')
        api.create_repo(HF_DATASET_REPO, repo_type='dataset', exist_ok=True)

        ds = hf_load('json', data_files=DATASET_PATH, split='train')
        ds.push_to_hub(HF_DATASET_REPO, token=HF_TOKEN)

        # Dataset card
        ds_card = """---
language:
- en
tags:
- safety
- women-safety
- indian-law
- bns
- nlp
license: cc-by-4.0
task_categories:
- text-classification
- text-generation
---

# NariRaksha Safety Dataset

A curated dataset of Indian women's safety scenarios with structured
assessments covering risk classification, severity, legal context
(BNS 2023 / IT Act), and recommended actions.

## Fields
| Field | Type | Description |
|---|---|---|
| `scenario` | str | The safety scenario narrative |
| `risk_type` | str | Risk category (e.g. cyberstalking) |
| `severity` | str | low / medium / high / critical |
| `reasoning` | str | Step-by-step safety analysis |
| `recommended_action` | str | Actionable advice for the victim |
| `legal_context` | str | Applicable BNS / IT Act sections |
| `confidence` | float | Assessment confidence score |
"""
        with open('/tmp/dataset_card.md', 'w') as f:
            f.write(ds_card)
        api.upload_file(
            path_or_fileobj='/tmp/dataset_card.md',
            path_in_repo='README.md',
            repo_id=HF_DATASET_REPO, repo_type='dataset', token=HF_TOKEN,
        )
        print(f'  ✅ Dataset published → https://huggingface.co/datasets/{HF_DATASET_REPO}')
    else:
        print('  ⚠️  HF_DATASET_REPO blank — skipping dataset publish.')

## 📈 Cell 10 — Evaluate (optional)
> Computes risk-type accuracy and severity accuracy on a held-out split.

In [ ]:
import json, random, torch
from nariraksha.data import SYSTEM_PROMPT
from unsloth import FastLanguageModel

FastLanguageModel.for_inference(model)

# Load a sample of test examples
with open(DATASET_PATH) as f:
    all_examples = [json.loads(l) for l in f if l.strip()]

test_sample = random.sample(all_examples, min(50, len(all_examples)))
print(f'Evaluating on {len(test_sample)} examples ...')

correct_risk, correct_sev, total = 0, 0, 0

for ex in test_sample:
    prompt = (
        f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
        f"<|im_start|>user\nAnalyse this safety scenario:\n\n{ex['scenario']}<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )
    inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=256, temperature=0.1, do_sample=True)
    resp = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    try:
        pred = json.loads(resp)
        if pred.get('risk_type', '').lower().strip() == ex['risk_type'].lower().strip():
            correct_risk += 1
        if pred.get('severity', '').lower().strip() == ex['severity'].lower().strip():
            correct_sev += 1
    except Exception:
        pass  # unparseable output
    total += 1

print(f'\n=== Evaluation Results ===')
print(f'Risk type accuracy : {correct_risk/total*100:.1f}% ({correct_risk}/{total})')
print(f'Severity accuracy  : {correct_sev/total*100:.1f}% ({correct_sev}/{total})')

# Save results
import json as _json
results = {'risk_accuracy': correct_risk/total, 'severity_accuracy': correct_sev/total, 'n': total}
with open(f'{OUTPUT_DIR}/eval_results.json', 'w') as f:
    _json.dump(results, f, indent=2)
print(f'\n✅ Results saved to {OUTPUT_DIR}/eval_results.json')